# Module 19 lab · Kernels

The code from every lesson of [this module](https://aicourses.dev/research/kernels), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**This lab needs a GPU.** Colab opens it on one; if not, choose Runtime → Change runtime type → T4 GPU.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 19.2 Triton from zero: vector add and fused softmax

Triton lets you write GPU kernels in Python. You start with adding two vectors, then merge the steps of softmax into one kernel, called fusion, so data is read from slow memory only once.

[Read the lesson](https://aicourses.dev/research/kernels/triton-from-zero)

### Programs, blocks and masks

Before writing it for the GPU, here is the same idea in plain NumPy, running the programs one after another:

In [ ]:
import numpy as np

def add_program(pid, x, y, out, BLOCK):
    offsets = pid * BLOCK + np.arange(BLOCK)  # this block's slots
    mask = offsets < len(x)                # the last block is short
    idx = offsets[mask]
    out[idx] = x[idx] + y[idx]
    return mask.sum()

n, BLOCK = 2500, 1024
x, y = np.arange(n, dtype=float), np.ones(n)
out = np.empty(n)
grid = (n + BLOCK - 1) // BLOCK                # like triton.cdiv
for pid in range(grid):                # a GPU runs these at once
    print(f"program {pid}: {add_program(pid, x, y, out, BLOCK)} items")
print(np.array_equal(out, x + y))
# program 0: 1024 items
# program 1: 1024 items
# program 2: 452 items
# True

### Vector add in Triton

Here is the same kernel for a GPU. The `@triton.jit` line tells Triton to compile the function into a GPU kernel the first time it runs. `x_ptr` and the others are **pointers**: the memory address where each tensor starts, so `x_ptr + offsets` is a whole block of addresses at once.

The lesson installs this first:

In [ ]:
!pip install triton

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(axis=0)               # which block is mine?
    offsets = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < n                        # the last block is short
    x = tl.load(x_ptr + offsets, mask=mask)   # from main memory
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x + y, mask=mask)

def add(x, y):
    out = torch.empty_like(x)
    n = out.numel()
    grid = (triton.cdiv(n, 1024),)            # number of programs
    add_kernel[grid](x, y, out, n, BLOCK=1024)
    return out

x = torch.rand(98_432, device="cuda")
y = torch.rand(98_432, device="cuda")
torch.testing.assert_close(add(x, y), x + y)
ms = triton.testing.do_bench(lambda: add(x, y))
print(3 * x.numel() * 4 / (ms * 1e-3) / 1e9, "GB/s")

### Fusing softmax

Softmax, from Module 6, turns a row of scores into probabilities. Written step by step in PyTorch, it is five operations, and each one is its own kernel with its own trip to main memory:

In [ ]:
def naive_softmax(x):
    x_max = x.max(dim=1).values           # read the row, write its max
    z = x - x_max[:, None]                # read both, write z
    numerator = torch.exp(z)              # read z, write exp
    denominator = numerator.sum(dim=1)    # read exp, write the sum
    return numerator / denominator[:, None]  # read both, write out

The Triton tutorial counts the traffic for an $M \times N$ input: $5MN + 2M$ numbers read and $3MN + 2M$ written. A fused kernel reads each number once and writes each answer once, $2MN$ in all:

In [ ]:
M, N = 4096, 1024                  # rows and columns
naive = (5 * M * N + 2 * M) + (3 * M * N + 2 * M)   # reads + writes
fused = M * N + M * N              # read once, write once
print(naive, fused, round(naive / fused, 3))
# 33570816 8388608 4.002

About four times less traffic, so a memory-bound fused kernel can be up to about four times faster, the theoretical speed-up the tutorial gives. Here is a fused softmax with one program per row:

In [ ]:
@triton.jit
def softmax_kernel(out_ptr, in_ptr, stride, n_cols,
                   BLOCK: tl.constexpr):
    row = tl.program_id(0)                # one program per row
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols
    x = tl.load(in_ptr + row * stride + cols, mask=mask,
                other=-float("inf"))      # read the row once
    x = x - tl.max(x, axis=0)             # every step on the chip
    num = tl.exp(x)
    y = num / tl.sum(num, axis=0)
    tl.store(out_ptr + row * stride + cols, y, mask=mask)

def softmax(x):
    rows, cols = x.shape
    out = torch.empty_like(x)
    BLOCK = triton.next_power_of_2(cols)  # the whole row fits
    softmax_kernel[(rows,)](out, x, x.stride(0), cols, BLOCK=BLOCK)
    return out

x = torch.randn(4096, 1000, device="cuda")
torch.testing.assert_close(softmax(x), torch.softmax(x, dim=1))

## 19.3 Tiled matmul and memory coalescing

A big matrix multiply is fastest when the GPU loads small blocks, called tiles, into its fast memory and reuses them. Tiling is the idea FlashAttention is built on.

[Read the lesson](https://aicourses.dev/research/kernels/tiled-matmul)

### One tile, many uses

where $n$ is the size of the square matrices and $T$ the tile size. Here it is in NumPy, counting every number fetched from the pretend main memory:

In [ ]:
import numpy as np

def tiled_matmul(A, B, T):
    n = A.shape[0]
    C = np.zeros((n, n))
    loads = 0                              # numbers fetched from "HBM"
    for i in range(0, n, T):               # one output tile each
        for j in range(0, n, T):
            acc = np.zeros((T, T))         # lives "on the chip"
            for k in range(0, n, T):
                a = A[i:i + T, k:k + T]    # load a tile of A
                b = B[k:k + T, j:j + T]    # load a tile of B
                loads += a.size + b.size
                acc += a @ b               # reuse each number T times
            C[i:i + T, j:j + T] = acc
    return C, loads

rng = np.random.default_rng(0)
n = 256
A, B = rng.normal(size=(n, n)), rng.normal(size=(n, n))
for T in (1, 8, 32, 128):
    C, loads = tiled_matmul(A, B, T)
    print(T, loads, np.allclose(C, A @ B))

### Coalescing: how warps read memory

If thread $k$ reads the 4-byte number at position $k$, the warp needs 128 bytes in a row: four transactions, no waste. If each thread reads every second number, the guide notes, half of every transaction is wasted and the load runs at half efficiency; the further apart the addresses, the worse it gets, until each thread needs a transaction of its own. Your CPU behaves the same way, because it too fetches memory in chunks of many bytes. Sum 4 million neighbouring numbers, then 4 million numbers spaced 16 apart:

In [ ]:
import time
import numpy as np

a = np.ones(64 * 2**20, dtype=np.float32)    # 256 MB of numbers
together = a[: 4 * 2**20]                     # 4M neighbours
spread = a[::16]                              # 4M, 64 bytes apart

def ms(x, repeats=20):
    x.sum()                                   # warm-up
    start = time.perf_counter()
    for _ in range(repeats):
        x.sum()
    return (time.perf_counter() - start) / repeats * 1000

print(f"neighbours: {ms(together):.2f} ms")
print(f"spread out: {ms(spread):.2f} ms")

### The matmul kernel in Triton

Here is a tiled matmul in Triton, written for clarity: one program per output tile on a two-dimensional grid, and sizes assumed to be multiples of the tile sizes. The official tutorial adds masks for any size, grouped ordering, and **autotuning**, which tries several tile sizes and keeps the fastest.

In [ ]:
@triton.jit
def matmul_kernel(a_ptr, b_ptr, c_ptr, K, sam, sak, sbk, sbn,
                  scm, scn, BM: tl.constexpr, BN: tl.constexpr,
                  BK: tl.constexpr):
    rm = tl.program_id(0) * BM + tl.arange(0, BM)   # my C rows
    rn = tl.program_id(1) * BN + tl.arange(0, BN)   # my C columns
    rk = tl.arange(0, BK)
    a_ptrs = a_ptr + rm[:, None] * sam + rk[None, :] * sak
    b_ptrs = b_ptr + rk[:, None] * sbk + rn[None, :] * sbn
    acc = tl.zeros((BM, BN), dtype=tl.float32)      # stays on chip
    for k in range(0, K, BK):                       # walk along K
        acc += tl.dot(tl.load(a_ptrs), tl.load(b_ptrs))
        a_ptrs += BK * sak                          # next A tile
        b_ptrs += BK * sbk                          # next B tile
    c_ptrs = c_ptr + rm[:, None] * scm + rn[None, :] * scn
    tl.store(c_ptrs, acc.to(tl.float16))

def matmul(a, b, BM=128, BN=128, BK=32):
    (M, K), N = a.shape, b.shape[1]
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    matmul_kernel[(M // BM, N // BN)](
        a, b, c, K, *a.stride(), *b.stride(), *c.stride(),
        BM=BM, BN=BN, BK=BK)
    return c

## 19.4 FlashAttention forward in Triton

FlashAttention computes attention tile by tile and never stores the full table of scores, using a running softmax. You write it yourself: the best-known case of research that wins by understanding the hardware.

[Read the lesson](https://aicourses.dev/research/kernels/flashattention-forward)

### The running softmax

Worked example: the scores 1, 3, 2, 5 in two chunks. After [1, 3]: $m = 3$, $\ell = e^{-2} + e^{0} = 1.1353$. Then [2, 5] brings $m' = 5$: $\ell' = 1.1353 \times e^{3-5} + e^{2-5} + e^{5-5} = 0.1536 + 0.0498 + 1 = 1.2034$. Check it against doing the whole row at once:

In [ ]:
import math

def online_sum(chunks):
    m, l = -math.inf, 0.0              # running max, running sum
    for chunk in chunks:
        m_new = max(m, max(chunk))
        l = l * math.exp(m - m_new)    # rescale what we had
        l += sum(math.exp(x - m_new) for x in chunk)
        m = m_new
        print(f"after {chunk}: max {m}, sum {l:.4f}")
    return m, l

online_sum([[1, 3], [2, 5]])
# after [1, 3]: max 3, sum 1.1353
# after [2, 5]: max 5, sum 1.2034
print(f"{sum(math.exp(x - 5) for x in [1, 3, 2, 5]):.4f}")  # 1.2034

### Tile by tile

In plain PyTorch, for one head, with $Q$, $K$ and $V$ of shape (sequence, head size):

In [ ]:
import math
import torch
import torch.nn.functional as F

def flash_forward(q, k, v, block=16):
    N, d = q.shape
    out, L = torch.empty_like(q), torch.empty(N)
    for i in range(0, N, block):              # one query block
        qi = q[i:i + block] / math.sqrt(d)
        m = torch.full((len(qi),), -math.inf)  # running max per row
        l = torch.zeros(len(qi))               # running sum per row
        acc = torch.zeros(len(qi), d)          # running output
        for j in range(0, N, block):          # one key/value block
            s = qi @ k[j:j + block].T          # a small score tile
            m_new = torch.maximum(m, s.max(dim=1).values)
            p = torch.exp(s - m_new[:, None])
            alpha = torch.exp(m - m_new)       # rescale the old
            l = l * alpha + p.sum(dim=1)
            acc = acc * alpha[:, None] + p @ v[j:j + block]
            m = m_new
        out[i:i + block] = acc / l[:, None]
        L[i:i + block] = m + torch.log(l)      # kept for backward
    return out, L

`L` is each row’s **logsumexp**, $m + \log \ell$, the log of softmax’s whole denominator. It is one number per query, and lesson 19.5 uses it to rebuild any tile of probabilities in the backward pass. Now compare with PyTorch’s own attention, `scaled_dot_product_attention`:

In [ ]:
torch.manual_seed(0)
q, k, v = torch.randn(3, 256, 64).unbind(0)
out, L = flash_forward(q, k, v)
ref = F.scaled_dot_product_attention(q, k, v)
print((out - ref).abs().max())     # tensor(5.3644e-07)
print(torch.allclose(L, torch.logsumexp(q @ k.T / 8, dim=1)))  # True

### The kernel in Triton

The Triton kernel is the same loop, with one program per query block and per head; the inner loop runs on the chip. This version assumes half-precision inputs of shape (batch, heads, sequence, head size), contiguous, with the sequence a multiple of the block sizes and no causal mask:

In [ ]:
@triton.jit
def flash_fwd(q_ptr, k_ptr, v_ptr, o_ptr, L_ptr, N, scale,
              D: tl.constexpr, BQ: tl.constexpr, BK: tl.constexpr):
    qb, bh = tl.program_id(0), tl.program_id(1)  # query block, head
    base = bh * N * D                             # this head's data
    rq = qb * BQ + tl.arange(0, BQ)
    rd = tl.arange(0, D)
    q = tl.load(q_ptr + base + rq[:, None] * D + rd[None, :])
    m = tl.full((BQ,), float("-inf"), tl.float32)
    l = tl.zeros((BQ,), tl.float32)
    acc = tl.zeros((BQ, D), tl.float32)
    for start in range(0, N, BK):                 # walk over keys
        rk = start + tl.arange(0, BK)
        kv = base + rk[:, None] * D + rd[None, :]
        k, v = tl.load(k_ptr + kv), tl.load(v_ptr + kv)
        s = tl.dot(q, tl.trans(k)) * scale        # one score tile
        m_new = tl.maximum(m, tl.max(s, axis=1))
        p = tl.exp(s - m_new[:, None])
        alpha = tl.exp(m - m_new)                 # rescale the old
        l = l * alpha + tl.sum(p, axis=1)
        acc = acc * alpha[:, None] + tl.dot(p.to(tl.float16), v)
        m = m_new
    out = (acc / l[:, None]).to(tl.float16)
    tl.store(o_ptr + base + rq[:, None] * D + rd[None, :], out)
    tl.store(L_ptr + bh * N + rq, m + tl.log(l))  # for backward

The launcher makes the outputs and the grid, one program per query block and per head:

In [ ]:
def flash_attention(q, k, v, BQ=64, BK=64):
    B, H, N, D = q.shape
    o = torch.empty_like(q)
    L = torch.empty((B * H, N), device=q.device, dtype=torch.float32)
    grid = (N // BQ, B * H)
    flash_fwd[grid](q, k, v, o, L, N, D ** -0.5, D=D, BQ=BQ, BK=BK)
    return o, L

## 19.5 FlashAttention backward

The backward pass recomputes parts of the forward pass instead of storing them. It is valuable depth but hard going, so it is optional and never blocks the project.

[Read the lesson](https://aicourses.dev/research/kernels/flashattention-backward)

### Tile by tile in PyTorch

With $P$ rebuilt from $L$ and $D$ known for every row, the backward pass runs over tiles like the forward pass. This version gives the outer loop to key and value blocks, so each $dK$ and $dV$ block is finished in one go, and adds each tile’s share into $dQ$:

In [ ]:
def flash_backward(q, k, v, out, L, dout, block=16):
    N, d = q.shape
    c = 1 / math.sqrt(d)
    D = (dout * out).sum(dim=1)             # one number per query
    dq, dk, dv = (torch.zeros_like(t) for t in (q, k, v))
    for j in range(0, N, block):            # one key/value block
        kj, vj = k[j:j + block], v[j:j + block]
        for i in range(0, N, block):        # every query block
            qi, doi = q[i:i + block], dout[i:i + block]
            p = torch.exp(c * qi @ kj.T - L[i:i + block, None])
            dv[j:j + block] += p.T @ doi    # rebuilt p, never stored
            dp = doi @ vj.T
            ds = p * (dp - D[i:i + block, None])
            dq[i:i + block] += c * ds @ kj
            dk[j:j + block] += c * ds.T @ qi
    return dq, dk, dv

Check it against PyTorch’s autograd, using `flash_forward` from lesson 19.4:

In [ ]:
torch.manual_seed(0)
q, k, v = torch.randn(3, 256, 64).unbind(0)
dout = torch.randn(256, 64)                 # gradient from above
out, L = flash_forward(q, k, v)             # from lesson 19.4
mine = flash_backward(q, k, v, out, L, dout)

for t in (q, k, v):
    t.requires_grad_()
F.scaled_dot_product_attention(q, k, v).backward(dout)
for name, a, t in zip("qkv", mine, (q, k, v)):
    print(name, f"{(a - t.grad).abs().max():.1e}")
# q 1.2e-06
# k 1.1e-06
# v 4.8e-07

## 19.6 Fused cross-entropy and optimizers

With a vocabulary of 100,000 tokens, the scores for every position fill memory fast. Fusing the loss into one kernel avoids storing them, and the same trick speeds up the optimizer.

[Read the lesson](https://aicourses.dev/research/kernels/fused-kernels)

### The logits nobody needs to keep

So the same cure works. Compute the logits a few positions at a time, use them, and throw them away. Done chunk by chunk, the loss is exactly the same:

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
tokens, width, vocab = 512, 256, 32_000
h = torch.randn(tokens, width)               # last hidden states
W = torch.randn(vocab, width) * 0.02         # output layer
y = torch.randint(0, vocab, (tokens,))       # the right next tokens

def chunked_loss(h, W, y, chunk=64):
    total = 0.0
    for i in range(0, len(h), chunk):        # a few tokens at a time
        logits = h[i:i + chunk] @ W.T        # only chunk x vocab
        total += F.cross_entropy(logits, y[i:i + chunk],
                                 reduction="sum")
    return total / len(h)

print(F.cross_entropy(h @ W.T, y).item())    # 10.429079055786133
print(chunked_loss(h, W, y).item())          # 10.429079055786133

Measure what autograd keeps, with the `saved_tensors_hooks` counter of lesson 18.4. This version counts each block of memory once, by its storage, since one tensor can be saved by two steps, and leaves out $h$ and $W$ themselves:

In [ ]:
from torch.utils.checkpoint import checkpoint
h.requires_grad_()
W.requires_grad_()
skip = {h.untyped_storage().data_ptr(), W.untyped_storage().data_ptr()}

def piece(i):                      # summed loss of tokens i to i + 63
    return F.cross_entropy(h[i:i + 64] @ W.T, y[i:i + 64],
                           reduction="sum")

def saved_mb(fn):                  # memory autograd keeps for backward
    kept = {}                      # one entry per block of memory
    def pack(t):
        s = t.untyped_storage()
        if s.data_ptr() not in skip:
            kept[s.data_ptr()] = s.nbytes()
        return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        fn()
    return round(sum(kept.values()) / 1e6, 1)

starts = range(0, tokens, 64)
print(saved_mb(lambda: F.cross_entropy(h @ W.T, y)))      # 65.5
print(saved_mb(lambda: sum(piece(i) for i in starts)))    # 65.5
print(saved_mb(lambda: sum(checkpoint(piece, i, use_reentrant=False)
                           for i in starts)))             # 0.0

### Fused optimizers

Both give the same answer as the simple loop, which you can check on Colab’s CPU runtime:

In [ ]:
import torch

def train(**opts):
    torch.manual_seed(0)
    model = torch.nn.Linear(64, 64)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-2, **opts)
    for _ in range(5):
        opt.zero_grad()
        model(torch.randn(16, 64)).pow(2).mean().backward()
        opt.step()
    return model.weight.detach()

loop = train(foreach=False)
fused = train(fused=True)
print(torch.allclose(loop, fused, atol=1e-6))    # True